# Goal
- Implementing Cost function for Logistic regression i.e. Log loss
- GD manual implementation for Log Reg
- Learning Curve
- Plotting Decision boundary
- Can do feature scaling if we want

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=2)  # reduced display precision on numpy arrays

In [ ]:
# Sigmoid
# Note: If z is a vector, exp will compute three values and we will get sigmoid for each element in the vector
def sigmoid(z):
    return 1 / (1 + np.exp(- z))

# Probability Predictions
def predict_probability(X, w, b):
    z = X @ w + b
    return sigmoid(z)

# Final predictions from probabilities
def predictions(X, w, b):
    probs = predict_probability(X, w, b)
    return (probs >= 0.5).astype(int)

In [ ]:
# Computing Logistic regression cost
def compute_cost_logistic(X, y, w, b):
    """
    Computes cost

    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters  
      b (scalar)       : model parameter
      
    Returns:
      cost (scalar): cost
    """

    m = len(y)

    sigm = predict_probability(X, w, b)
    cost = - np.sum(y * np.log(sigm) + (1 - y) * np.log(1 - sigm)) / m

    return cost

In [ ]:
# Gradients computation
def compute_gradients(X, y, w, b):
    m = len(y)
    f_wb = predict_probability(X, w, b)
    err = f_wb - y

    dj_dw = (X.T @ err) / m
    dj_db = np.sum(err) / m

    return dj_dw, dj_db

In [ ]:
# Gradient descent implementation with history
def gradient_descent(
    X, y, w_init, b_init,
    alpha, iterations,
    record_factor,
    stopping_criteria = 1e-1
    ):

    w, b = w_init.copy(), b_init

    history ={
        'iteration': [0],
        'cost': [compute_cost_logistic(X, y, w, b)],
        'w': [w.copy()],
        'b': [b]
    }

    for itr in range(iterations):

        dj_dw, dj_db = compute_gradients(X, y, w, b)

        w -= alpha * dj_dw
        b -= alpha * dj_db
        
        cost = compute_cost_logistic(X, y, w, b)

        if (itr + 1) % record_factor == 0:
            history["iteration"].append(itr + 1)
            history["cost"].append(cost)
            history["w"].append(w.copy())
            history["b"].append(b)

        if cost < stopping_criteria: # Stop when cost becomes sufficiently small
            if history["iteration"][-1] != itr + 1:
                history["iteration"].append(itr + 1)
                history["cost"].append(cost)
                history["w"].append(w.copy())
                history["b"].append(b)
            break

    return w, b, history

In [ ]:
# Plot Learning curve
def plot_learning_curve(history):
    cost = history['cost']
    iterations = history['iteration']

    plt.grid(True)
    plt.tight_layout()
    plt.plot(iterations, cost)
    plt.xlabel('Iterations')
    plt.ylabel('J(w, b)')
    plt.title('Learning Curve')
    plt.show()

# Plot Decision boundary
def plot_decision_boundary(X, y, w, b):
    plt.scatter(X[y == 0, 0], X[y == 0, 1], label="Class 0", marker='o', c='b')
    plt.scatter(X[y == 1, 0], X[y == 1, 1], label="Class 1", marker='x', c='r')

    x1 = np.linspace(X[:, 0].min(), X[:, 0].max(), 100)
    x2 = -(w[0] * x1 + b) / w[1]

    plt.plot(x1, x2, 'g-', label="Decision Boundary") # g-: means green line that is solid dash (-)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend()
    plt.show()

In [ ]:
# Training samples
X_train = np.array([
        [0.5, 1.5],
        [1, 1],
        [1.5, 0.5],
        [3, 0.5],
        [2, 2],
        [1, 2.5]
    ])  #(m,n)
y_train = np.array([0, 0, 0, 1, 1, 1]) #(m,)

In [ ]:
w_init, b_init = np.array([0., 0.]), 0
iterations = 10000
alpha = 0.1
record_factor = 1000
stopping_criteria = 1e-3
w_fin, b_fin, history = gradient_descent(X_train, y_train, w_init, b_init, alpha, iterations, record_factor, stopping_criteria)

plot_decision_boundary(X_train, y_train, w_fin, b_fin)
plot_learning_curve(history)
print(f"\nFinal weights: {w_fin}")
print(f"Final Bias: {b_fin}")

preds = predictions(X_train, w_fin, b_fin)
accuracy = np.mean(preds == y_train)

print(f"Predictions: {preds}")
print(f"Actual Targets: {y_train}")
print(f"Accuracy: {accuracy * 100:.2f}%")